# 🌲 YOLOv8-seg 樹幹偵測模型訓練（萬張級資料 + 學術加權 + GPU 優化版）

**目標**：訓練一個只辨識 `tree_trunk`（樹幹）的實例分割模型

**資料**：合併 6 個 Roboflow 資料集 + **Urban Street Trunk**（7,675 張, 2x）+ **Xiang et al.**（~235 張, 3x）→ 有效 **~18,500+ 張**，學術資料加權但保留 Roboflow 多樣性

**驗證**：Xiang et al. 獨立驗證子集（~59 張, iPhone 近距離, SAM 標註）→ 獨立分割品質檢驗

**續練**：支援從你現有的 `.pt` 模型繼續訓練（fine-tune on top of fine-tune）

**匯出**：TFLite INT8 (~6-12MB) → Flutter 手機端 30+ FPS

---

## 使用前設定
1. 上方選單 → **執行階段** → **變更執行階段類型** → 選 **GPU**
   - H100 (~79GB) → 訓練約 25-40 分鐘 🚀 推薦（算力最猛）
   - A100 (40/80GB) → 訓練約 35-50 分鐘 ✅ 很夠用
   - L4 (24GB) → 訓練約 60-90 分鐘
   - T4 (16GB) → 訓練約 120-180 分鐘

2. 按照 Cell 順序從上到下執行3. 最後下載匯出的模型檔

## Step 0：確認 GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    total = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', None)
    if total:
        print(f'VRAM: {total / 1024**3:.1f} GB')
    else:
        print(f'VRAM: (see nvidia-smi output above)')

## Step 1：安裝依賴

In [ ]:
!pip install ultralytics roboflow supervision -q
print('✓ 安裝完成')

## Step 1.5：上傳你現有的 .pt 模型（續練用）

如果你已經有訓練過的模型（如 `tree_trunk_seg_best.pt`），上傳它來**繼續訓練**。
這樣模型不用從零學起，直接在你已有的成果上精進。

- ✅ 有現有模型 → 上傳後繼續訓練
- ⭕ 第一次訓練 → 跳過此步驟，會自動用 COCO 預訓練權重

In [ ]:
#@title 上傳現有模型 { display-mode: "form" }
import os

#@markdown ### 選擇模式
TRAINING_MODE = 'fresh' #@param ['resume', 'fresh'] {type:"string"}
#@markdown - **resume**：上傳你的 .pt 模型，基於它繼續訓練
#@markdown - **fresh**：從 COCO 預訓練權重重新開始

#@markdown ---
#@markdown ### 自動續練（Crash Recovery）
AUTO_RESUME_FROM_CRASH = True #@param {type:"boolean"}
#@markdown - **True**：如果偵測到上次 crash 留下的 `last.pt`，自動載入那些權重繼續練
#@markdown - **False**：每次都從 COCO 預訓練重新來

RESUME_MODEL_PATH = None

if TRAINING_MODE == 'resume':
    from google.colab import files
    print('📁 請上傳你的 .pt 模型檔（例如 tree_trunk_seg_best.pt）...')
    print('   如果沒有，請把上方 TRAINING_MODE 改為 "fresh"\n')
    try:
        uploaded = files.upload()
        for filename, data in uploaded.items():
            save_path = f'/content/{filename}'
            with open(save_path, 'wb') as f:
                f.write(data)
            RESUME_MODEL_PATH = save_path
            size_mb = os.path.getsize(save_path) / 1024 / 1024
            print(f'\n✓ 已上傳: {filename} ({size_mb:.1f} MB)')

            # 驗證是否為有效的 YOLO 模型
            from ultralytics import YOLO
            try:
                test_model = YOLO(save_path)
                print(f'  模型類型: {test_model.task}')
                print(f'  類別數: {test_model.model.nc if hasattr(test_model.model, "nc") else "N/A"}')
                print(f'\n✅ 模型驗證成功！將基於此模型繼續訓練')
                del test_model
            except Exception as e:
                print(f'⚠️ 模型載入警告: {e}')
                print('  將嘗試繼續使用...')
    except Exception as e:
        print(f'上傳取消或失敗: {e}')
        print('將改用 COCO 預訓練權重（fresh mode）')
        TRAINING_MODE = 'fresh'
else:
    print('✓ Fresh mode：將從 COCO 預訓練權重開始訓練')
    print('  （如果想續練，把 TRAINING_MODE 改為 "resume"）')
    if AUTO_RESUME_FROM_CRASH:
        print('  💡 AUTO_RESUME_FROM_CRASH = True：若上次有 crash checkpoint 會自動偵測')

## Step 2：設定並下載 Roboflow 資料集

合併以下 CC BY 4.0 開源資料集（Roboflow 部分）：

| # | 資料集 | 圖片數 | 類型 | 說明 |
|---|--------|--------|------|------|
| 1 | tree-trunks/tree-trunk-detection-bi-axe v1 | 1,302 | Instance Seg | 主資料集，trunk/post/sprinkler |
| 2 | tree-trunks/cherry-trunks v2 | 337 | Instance Seg | 櫻桃樹幹，增加多樣性 |
| 3 | imageprocessing-mo6fy/tree_trunk-weppw v1 | 1,092 | Instance Seg | 6 classes：trunk/0/2=樹幹，1/post/sprinkler=排除 |
| 4 | projetosia/trunk-tree-z1bpo v1 | 140 | Instance Seg | 巴西樹幹 |
| 5 | wurdataset/tree-trunk-segmentation-ixblx v1 | 172 | Instance Seg | 精確標註小型資料集 |
| **6** | **Urban Street Trunk (Kaggle)** | **7,675** | **Instance Seg** | **🎓 學術級，29 樹種，Step 2.5 下載** |
| **7** | **Xiang et al. (Zenodo)** | **294** | **Seg (SAM)** | **🎓 80% 加入訓練 (3x) + 20% 獨立驗證，Step 2.7** |

Roboflow 合併後自動：
- **嚴格過濾**：class name 必須含 `trunk` 才保留（不用 `tree` 避免誤配整棵樹/樹冠）
- **手動確認**：weppw 資料集的 class `0` 和 `2` 經確認也是樹幹，額外保留
- 多類別資料集無 `trunk` → 整個跳過（不像之前的全保留 fallback）
- 單類別資料集：合理假設唯一 class = trunk

- 影像去重（快速 hash：檔案大小 + 前 2KB）- Urban Street 資料在 Step 2.5 另外下載 & 轉換

In [ ]:
#@title 設定 Roboflow API Key & 資料集清單 { display-mode: "form" }

#@markdown ### Roboflow 設定
#@markdown 前往 https://app.roboflow.com/settings/api 取得 Private API Key
ROBOFLOW_API_KEY = 'uwNljxzf8xgKGZ9Is3py' #@param {type:"string"}

# ===== 要合併的資料集清單 =====
# 每個 tuple = (workspace, project, version, description)
DATASETS = [
    # --- 核心資料集（大而全） ---
    ('tree-trunks', 'tree-trunk-detection-bi-axe', 1, '🌳 主資料集 1.3k 張，trunk/post/sprinkler'),
    ('tree-trunks', 'cherry-trunks', 2, '🌸 櫻桃樹幹 337 張，增加多樣性'),
    ('imageprocessing-mo6fy', 'tree_trunk-weppw', 1, '🌲 6 classes, trunk/0/2=樹幹, 排除 1/post/sprinkler'),

    # --- 補充資料集（增加場景多樣性） ---
    ('projetosia', 'trunk-tree-z1bpo', 1, '🇧🇷 巴西樹幹 140 張'),
    ('wurdataset', 'tree-trunk-segmentation-ixblx', 1, '🎯 精確標註 172 張'),
]

if not ROBOFLOW_API_KEY:
    print('❌ 請填入 Roboflow API Key！')
    print('   前往 https://app.roboflow.com/settings/api 取得')
else:
    print(f'✓ API Key 已設定')
    print(f'\n將下載 {len(DATASETS)} 個資料集：')
    for i, (ws, proj, ver, desc) in enumerate(DATASETS, 1):
        print(f'  {i}. {ws}/{proj} v{ver} — {desc}')

In [ ]:
# ===== 下載所有資料集 =====
from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

downloaded_datasets = []
for i, (ws, proj, ver, desc) in enumerate(DATASETS, 1):
    print(f'\n{"="*50}')
    print(f'[{i}/{len(DATASETS)}] 下載 {ws}/{proj} v{ver}...')
    try:
        project = rf.workspace(ws).project(proj)
        ds = project.version(ver).download(
            model_format='yolov8',
            location=f'/content/dataset_{i}',
            overwrite=True
        )
        # 統計
        for split in ['train', 'valid', 'test']:
            img_dir = os.path.join(ds.location, split, 'images')
            if os.path.exists(img_dir):
                count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
                print(f'  {split}: {count} images')
        downloaded_datasets.append(ds)
        print(f'  ✓ 完成')
    except Exception as e:
        print(f'  ✗ 下載失敗: {e}')
        print(f'  跳過此資料集，繼續...')

print(f'\n{"="*50}')
print(f'✓ 成功下載 {len(downloaded_datasets)}/{len(DATASETS)} 個資料集')

## Step 2.5：下載 Urban Street Trunk 學術級資料集（7,675 張）

🎓 **Urban Street: Trunk Detection** — 浙江農林大學（SCI 論文，GNU LGPL 3.0）

| 項目 | 說明 |
|------|------|
| 圖片數 | **7,675** 張高解析度街道實景 |
| 類別 | 29 種城市行道樹樹幹（**全部對應到 `tree_trunk` 類別**） |
| 格式 | VOC2012 Bitmap Mask（SegmentationClass PNG） → contour → YOLO-seg |
| 來源 | 手機拍攝城市街道樹木 ← 非常符合 App 使用場景！ |
| 論文 | *Urban Street Tree Dataset for Image Classification and Instance Segmentation* |

加上 Roboflow 的 5 個資料集，**總計 ~10,000+ 張標註影像** 🔥

In [ ]:
#@title 下載 Urban Street Trunk 資料集（從 Kaggle） { display-mode: "form" }

#@markdown ### Kaggle 設定
#@markdown 前往 https://www.kaggle.com/settings → API → Create New Token 下載 kaggle.json
#@markdown
#@markdown **或**直接填入帳號與 Key：
KAGGLE_USERNAME = 'liuminhhao' #@param {type:"string"}
KAGGLE_KEY = '008487131ccae0319171497f4f54a7c5'      #@param {type:"string"}

import os

# ===== 設定 Kaggle 認證 =====
kaggle_configured = False

# 方法1：直接填入帳號密碼
if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY
    kaggle_configured = True
    print(f'✓ 使用填入的 Kaggle 帳號: {KAGGLE_USERNAME}')
else:
    # 方法2：上傳 kaggle.json
    kaggle_json_path = os.path.expanduser('~/.kaggle/kaggle.json')
    if os.path.exists(kaggle_json_path):
        kaggle_configured = True
        print(f'✓ 找到已有的 kaggle.json')
    else:
        print('📤 請上傳 kaggle.json...')
        try:
            from google.colab import files
            uploaded = files.upload()
            if 'kaggle.json' in uploaded:
                os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
                with open(kaggle_json_path, 'wb') as f:
                    f.write(uploaded['kaggle.json'])
                os.chmod(kaggle_json_path, 0o600)
                kaggle_configured = True
                print('✓ kaggle.json 上傳成功')
        except Exception as e:
            print(f'⚠️ 上傳失敗: {e}')

# ===== 下載資料集 =====
URBAN_STREET_DIR = '/content/urban_street_trunk'

if kaggle_configured:
    # 安裝 kaggle CLI
    os.system('pip install -q kaggle')

    # 下載 Urban Street 資料集
    DATASET_SLUG = 'erickendric/tree-dataset-of-urban-street-segmentation-trunk'
    print(f'\n📥 下載 {DATASET_SLUG}...')
    print('   (約 6-13 GB，請稍候...)')

    ret = os.system(f'kaggle datasets download -d {DATASET_SLUG} -p {URBAN_STREET_DIR} --unzip')

    if ret == 0:
        # 統計下載檔案
        total_files = 0
        for root, dirs, fnames in os.walk(URBAN_STREET_DIR):
            total_files += len(fnames)
        print(f'\n✓ 下載完成！共 {total_files} 個檔案')

        # 列出目錄結構
        print(f'\n目錄結構：')
        for item in sorted(os.listdir(URBAN_STREET_DIR)):
            item_path = os.path.join(URBAN_STREET_DIR, item)
            if os.path.isdir(item_path):
                sub_count = len(os.listdir(item_path))
                print(f'  📁 {item}/ ({sub_count} items)')
            else:
                size_mb = os.path.getsize(item_path) / 1024 / 1024
                print(f'  📄 {item} ({size_mb:.1f} MB)')
    else:
        print('❌ 下載失敗！請確認 Kaggle 認證')
        print(f'   前往 https://www.kaggle.com/datasets/{DATASET_SLUG}')
        print('   按 "Accept" 同意使用條款後重新執行此 Cell')
else:
    print('❌ 未設定 Kaggle 認證')
    print('   方法1: 填入上方的 KAGGLE_USERNAME 和 KAGGLE_KEY')
    print('   方法2: 上傳 kaggle.json（前往 kaggle.com/settings → API → Create New Token）')

In [ ]:
#@title VOC Bitmap Mask → YOLO-seg 格式轉換（Urban Street） { display-mode: "form" }
import os, shutil, cv2
import numpy as np
from pathlib import Path

URBAN_STREET_DIR = '/content/urban_street_trunk'
URBAN_CONVERTED_DIR = '/content/urban_street_converted'

# ===== 自動找到 VOC 根目錄 =====
voc_root = None
for root, dirs, files in os.walk(URBAN_STREET_DIR):
    if 'JPEGImages' in dirs and 'SegmentationClass' in dirs:
        voc_root = root
        break

if voc_root is None:
    raise RuntimeError(f'❌ 找不到 VOC 資料集結構 (JPEGImages + SegmentationClass)\n'
                       f'   請檢查 {URBAN_STREET_DIR} 目錄')

print(f'✓ 找到 VOC 根目錄: {voc_root}')

JPEG_DIR = os.path.join(voc_root, 'JPEGImages')
SEG_DIR  = os.path.join(voc_root, 'SegmentationClass')
SPLIT_DIR = os.path.join(voc_root, 'ImageSets', 'Segmentation')
CLASS_NAMES_FILE = os.path.join(voc_root, 'class_names.txt')

# ===== 讀取 class_names.txt =====
if os.path.exists(CLASS_NAMES_FILE):
    with open(CLASS_NAMES_FILE, 'r') as f:
        class_names = [line.strip() for line in f if line.strip()]
    print(f'\n📋 class_names.txt ({len(class_names)} 類):')
    for i, name in enumerate(class_names):
        print(f'   [{i}] {name}')
else:
    class_names = []
    print('⚠️ 找不到 class_names.txt，將使用所有非背景像素')

# ===== 統計檔案 =====
jpeg_files = [f for f in os.listdir(JPEG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
seg_files  = [f for f in os.listdir(SEG_DIR) if f.lower().endswith('.png')]

print(f'\n📊 資料統計:')
print(f'   JPEGImages/: {len(jpeg_files)} 張')
print(f'   SegmentationClass/: {len(seg_files)} 張 masks')

# 讀取 train/val/test split
splits = {}
if os.path.exists(SPLIT_DIR):
    for split_file in os.listdir(SPLIT_DIR):
        if split_file.endswith('.txt'):
            split_name = split_file.replace('.txt', '')
            with open(os.path.join(SPLIT_DIR, split_file), 'r') as f:
                ids = [line.strip() for line in f if line.strip()]
            splits[split_name] = set(ids)
            print(f'   {split_name}: {len(ids)} 張')

# ===== 清理舊轉換 =====
if os.path.exists(URBAN_CONVERTED_DIR):
    shutil.rmtree(URBAN_CONVERTED_DIR)
os.makedirs(os.path.join(URBAN_CONVERTED_DIR, 'images'), exist_ok=True)
os.makedirs(os.path.join(URBAN_CONVERTED_DIR, 'labels'), exist_ok=True)

# ===== Bitmap mask → YOLO polygon 轉換 =====
def mask_to_yolo_polygons(mask_path, img_w, img_h, min_area=100):
    """
    讀取 VOC SegmentationClass PNG mask，提取輪廓轉成 YOLO polygon。
    VOC mask: 像素值 = class_id (0=background, 255=border/ignore)
    所有非背景類別都映射到 class 0 (tree_trunk)。
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        # 嘗試用彩色讀取再轉灰度
        mask_bgr = cv2.imread(mask_path, cv2.IMREAD_COLOR)
        if mask_bgr is None:
            return []
        # VOC 可能用 palette PNG，直接讀 palette index
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        if mask is None:
            return []
        if len(mask.shape) == 3:
            # 彩色 mask → 需要用調色盤反查 class id
            # 簡化處理：任何非黑色像素 = 前景(trunk)
            gray = cv2.cvtColor(mask_bgr, cv2.COLOR_BGR2GRAY)
            mask = (gray > 0).astype(np.uint8) * 255

    yolo_lines = []

    # 建立前景 binary mask（排除 0=background 和 255=border/void）
    if mask.max() <= 1:
        # 已經是 binary
        binary = (mask > 0).astype(np.uint8) * 255
    else:
        # VOC 格式: 0=bg, 1-254=classes, 255=void/border
        binary = np.zeros_like(mask)
        binary[(mask > 0) & (mask < 255)] = 255

    # 找輪廓
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_L1)

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue

        # 簡化輪廓（減少點數但保持形狀）
        epsilon = 0.001 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)

        if len(approx) < 3:
            continue

        # 正規化座標到 [0, 1]
        points = []
        for pt in approx:
            x = pt[0][0]
            y = pt[0][1]
            nx = max(0.0, min(1.0, x / img_w))
            ny = max(0.0, min(1.0, y / img_h))
            points.extend([f'{nx:.6f}', f'{ny:.6f}'])

        yolo_lines.append('0 ' + ' '.join(points))

    return yolo_lines


# ===== 批量轉換 =====
converted = 0
skipped_no_mask = 0
skipped_empty = 0
errors = 0

# 取得所有 mask 檔案
mask_files = sorted([f for f in os.listdir(SEG_DIR) if f.lower().endswith('.png')])
total = len(mask_files)

print(f'\n🔄 開始轉換 {total} 張 masks...')

for i, mask_fname in enumerate(mask_files):
    if (i + 1) % 1000 == 0 or (i + 1) == total:
        print(f'  進度: {i+1}/{total} ({converted} converted)')

    mask_path = os.path.join(SEG_DIR, mask_fname)
    stem = os.path.splitext(mask_fname)[0]

    # 找到對應的圖片
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG']:
        candidate = os.path.join(JPEG_DIR, stem + ext)
        if os.path.exists(candidate):
            img_path = candidate
            break

    if img_path is None:
        skipped_no_mask += 1
        continue

    # 讀取圖片尺寸（用 PIL 只讀 header，不解碼像素，快 100x）
    from PIL import Image as _PILImage
    try:
        with _PILImage.open(img_path) as _pil_img:
            img_w, img_h = _pil_img.size
    except Exception:
        errors += 1
        continue

    # 轉換 mask → YOLO polygons
    try:
        yolo_lines = mask_to_yolo_polygons(mask_path, img_w, img_h)
    except Exception as e:
        errors += 1
        continue

    if not yolo_lines:
        skipped_empty += 1
        continue

    # 輸出
    img_ext = os.path.splitext(img_path)[1]
    out_name = f'urban_{converted:05d}'
    dst_img = os.path.join(URBAN_CONVERTED_DIR, 'images', out_name + img_ext)
    dst_lbl = os.path.join(URBAN_CONVERTED_DIR, 'labels', out_name + '.txt')
    # 用 hard link 代替複製（瞬間完成，不佔額外磁碟空間）
    try:
        os.link(img_path, dst_img)
    except OSError:
        shutil.copy2(img_path, dst_img)  # fallback
    with open(dst_lbl, 'w') as f:
        f.write('\n'.join(yolo_lines) + '\n')
    converted += 1

# ===== 結果統計 =====
out_imgs = len(os.listdir(os.path.join(URBAN_CONVERTED_DIR, 'images')))
out_lbls = len(os.listdir(os.path.join(URBAN_CONVERTED_DIR, 'labels')))

print(f'\n{"="*50}')
print(f'✓ Bitmap Mask → YOLO-seg 轉換完成！')
print(f'  成功轉換:      {converted} 張')
print(f'  找不到圖片:    {skipped_no_mask}')
print(f'  空 mask 跳過:  {skipped_empty}')
print(f'  解析錯誤:      {errors}')
print(f'\n  輸出目錄: {URBAN_CONVERTED_DIR}/')
print(f'  images/: {out_imgs} 張')
print(f'  labels/: {out_lbls} 個')
print(f'{"="*50}')

# 顯示範例標註
if converted > 0:
    print(f'\n📝 範例 YOLO 標註 (前 3 個):')
    lbl_dir = os.path.join(URBAN_CONVERTED_DIR, 'labels')
    for lbl_f in sorted(os.listdir(lbl_dir))[:3]:
        with open(os.path.join(lbl_dir, lbl_f)) as f:
            lines = f.readlines()
        print(f'  {lbl_f}: {len(lines)} 個 polygon')

        for line in lines[:2]:
            parts = line.strip().split()
            n_pts = (len(parts) - 1) // 2
            print(f'    class={parts[0]}, {n_pts} points, first_pt=({parts[1]},{parts[2]})')

## Step 2.7：下載 Xiang et al. 樹木資料集（Zenodo，294 張真實樹木 → 80% 訓練 / 20% 驗證）

🎓 **學術級資料** — 294 張真實樹木，80% 加入訓練（3x 加權），20% 保留為獨立驗證集。

| 項目 | 說明 |
|------|------|
| 圖片數 | 294 張（iPhone 13 Pro 拍攝，近距離 0.25~5m 真實樹木） |
| 標註 | SAM 生成的 binary segmentation masks（`-tm.jpg` 後綴） |
| 解析度 | 1920×1440（直式拍攝，樹幹佔畫面 9%~63%） |
| 格式 | Zenodo ZIP → `tree/treeRGB` + `tree/treeSeg` → 自動轉 YOLO-seg polygon |
| DOI | `10.5281/zenodo.10650629` |
| 分割 | 80% (~235 張) 加入 Step 3 訓練合併 / 20% (~59 張) Step 6.5 獨立驗證 |
| 論文 | Xiang et al. (2025) Sensors, 25(16), 5060 |

**資料夾結構**：
- `tree/treeRGB/` — RGB 影像（訓練輸入）
- `tree/treeSeg/` — SAM 分割 mask（訓練標註）
- `tree/treePNG/` — iPhone LiDAR 深度圖（260×192, 16-bit → 公尺）⬅️ **本流程不使用**，為論文的 LiDAR 測量管線資料
- `tree/tree_log.csv` — 每棵樹的真實直徑 (TD)、拍攝距離 (CapDis)、相機內參等

📌 ZIP 中的 `cylinder/` 資料夾為白色壓克力圓柱體（非樹木），已自動忽略。

In [ ]:
#@title 下載 Xiang et al. 驗證集（Zenodo） { display-mode: "form" }
import os, glob

XIANG_DIR = '/content/xiang_validation'
os.makedirs(XIANG_DIR, exist_ok=True)

print('📥 下載 Xiang et al. 分割驗證集...')
print('   DOI: 10.5281/zenodo.10650629')
print('   (~100+ MB，請稍候...)\n')

# 方法1: zenodo_get
os.system('pip install -q zenodo_get')
ret = os.system(f'cd {XIANG_DIR} && zenodo_get 10.5281/zenodo.10650629')

if ret != 0:
    # 方法2: 直接 wget
    print('⚠️ zenodo_get 失敗，嘗試直接下載...')
    ZIP_URL = 'https://zenodo.org/records/10650629/files/data%20and%20code.zip?download=1'
    ret = os.system(f'wget -q --show-progress -O "{XIANG_DIR}/data_and_code.zip" "{ZIP_URL}"')

# 解壓所有 zip
zips = glob.glob(os.path.join(XIANG_DIR, '*.zip'))
for z in zips:
    print(f'  解壓: {os.path.basename(z)}')
    os.system(f'unzip -q -o "{z}" -d "{XIANG_DIR}"')

# 列出目錄結構
print(f'\n📁 目錄結構：')
for root, dirs, files in os.walk(XIANG_DIR):
    level = root.replace(XIANG_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/ ({len(files)} files)')
    if level < 3:
        for f in sorted(files)[:5]:
            print(f'{indent}  📄 {f}')
        if len(files) > 5:
            print(f'{indent}  ... 和 {len(files)-5} 個其他檔案')

print(f'\n✓ Xiang et al. 驗證集下載完成！')

In [ ]:
#@title 轉換 Xiang et al. Binary Masks → YOLO-seg（80% 訓練 / 20% 驗證） { display-mode: "form" }
import cv2, yaml
import numpy as np
import os, shutil, random
from pathlib import Path

XIANG_DIR = '/content/xiang_validation'
XIANG_TRAIN = '/content/xiang_yolo_train'        # 80% → Step 3 訓練合併
XIANG_VAL = '/content/xiang_yolo_validation'      # 20% → Step 6.5 獨立驗證
TRAIN_RATIO = 0.8
SPLIT_SEED = 42

# 清理舊輸出
for d in [XIANG_TRAIN, XIANG_VAL]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(os.path.join(d, 'images'), exist_ok=True)
    os.makedirs(os.path.join(d, 'labels'), exist_ok=True)

# ===== 搜尋 tree/ 資料夾中的 treeRGB + treeSeg =====
# ZIP 結構: data and code/ → tree/ → treeRGB/, treeSeg/
# (忽略 cylinder/ 資料夾 — 那是白色壓克力圓柱體，非真實樹木)
rgb_dirs = list(Path(XIANG_DIR).rglob('treeRGB'))
seg_dirs = list(Path(XIANG_DIR).rglob('treeSeg'))

if not rgb_dirs or not seg_dirs:
    raise FileNotFoundError(
        f'❌ 找不到 tree/treeRGB 或 tree/treeSeg 資料夾！\n'
        f'   請確認 {XIANG_DIR} 中有 data and code/tree/ 目錄結構'
    )

rgb_dir = rgb_dirs[0]
seg_dir = seg_dirs[0]
print(f'📂 RGB 目錄: {rgb_dir}')
print(f'📂 Seg 目錄: {seg_dir}')

rgb_files = sorted([f for f in rgb_dir.iterdir()
                    if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp')])
print(f'   找到 {len(rgb_files)} 張 RGB 影像')

converted = 0
skipped_no_mask = 0
skipped_no_contour = 0
mask_coverage_pcts = []
converted_pairs = []  # [(rgb_path, yolo_lines)]

for rgb_path in rgb_files:
    stem = rgb_path.stem  # e.g. rgb-UUID-timestamp

    # 搜尋對應 mask（treeSeg 中的 -tm.jpg 後綴）
    possible_masks = [
        seg_dir / f'{stem}-tm.jpg', seg_dir / f'{stem}-tm.png',
        seg_dir / f'{stem}_mask.jpg', seg_dir / f'{stem}_mask.png',
        seg_dir / f'{stem}.jpg', seg_dir / f'{stem}.png',
    ]
    mask_path = next((mp for mp in possible_masks if mp.exists()), None)
    if mask_path is None:
        skipped_no_mask += 1
        continue

    # 讀取 mask
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        skipped_no_mask += 1
        continue

    # 二值化 & 找輪廓
    # SAM 生成的 mask 是 JPEG 格式，有壓縮雜訊 → threshold 去除灰階邊緣
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        skipped_no_contour += 1
        continue

    H, W = mask.shape[:2]
    mask_px = (binary > 0).sum()
    mask_coverage_pcts.append(mask_px / (H * W) * 100)

    yolo_lines = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 500:  # 過濾 JPEG 壓縮雜訊產生的小碎片
            continue
        epsilon = 0.005 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        if len(approx) < 3:
            continue

        points = []
        for pt in approx:
            x = max(0.0, min(1.0, pt[0][0] / W))
            y = max(0.0, min(1.0, pt[0][1] / H))
            points.extend([f'{x:.6f}', f'{y:.6f}'])
        yolo_lines.append('0 ' + ' '.join(points))

    if yolo_lines:
        converted_pairs.append((rgb_path, yolo_lines))
        converted += 1

print(f'  成功轉換: {converted} / {len(rgb_files)} 張\n')

# ===== 隨機分割 train / val =====
random.seed(SPLIT_SEED)
indices = list(range(len(converted_pairs)))
random.shuffle(indices)
n_train = int(len(indices) * TRAIN_RATIO)

train_count = 0
val_count = 0
for rank, idx in enumerate(indices):
    rgb_path, yolo_lines = converted_pairs[idx]
    is_train = rank < n_train
    out_dir = XIANG_TRAIN if is_train else XIANG_VAL
    out_idx = train_count if is_train else val_count
    out_name = f'xiang_tree_{out_idx:04d}'
    ext = rgb_path.suffix

    shutil.copy2(str(rgb_path), os.path.join(out_dir, 'images', out_name + ext))
    with open(os.path.join(out_dir, 'labels', out_name + '.txt'), 'w') as f:
        f.write('\n'.join(yolo_lines) + '\n')

    if is_train:
        train_count += 1
    else:
        val_count += 1

# data.yaml — 驗證用（Step 6.5 讀這個）
with open(os.path.join(XIANG_VAL, 'data.yaml'), 'w') as f:
    yaml.dump({
        'path': XIANG_VAL,
        'val': 'images',
        'nc': 1,
        'names': ['tree_trunk']
    }, f, default_flow_style=False)

# data.yaml — 訓練用參考
with open(os.path.join(XIANG_TRAIN, 'data.yaml'), 'w') as f:
    yaml.dump({
        'path': XIANG_TRAIN,
        'train': 'images',
        'nc': 1,
        'names': ['tree_trunk']
    }, f, default_flow_style=False)

print(f'{"="*50}')
print(f'✓ Xiang et al. 樹木資料轉換完成！')
print(f'  📚 訓練集: {train_count} 張 → {XIANG_TRAIN}/')
print(f'  🎯 驗證集: {val_count} 張 → {XIANG_VAL}/')
if skipped_no_mask:
    print(f'  ⚠️ 跳過（無對應 mask）: {skipped_no_mask}')
if skipped_no_contour:
    print(f'  ⚠️ 跳過（無有效輪廓）: {skipped_no_contour}')
if mask_coverage_pcts:
    print(f'  Mask 覆蓋率: {np.min(mask_coverage_pcts):.1f}% ~ {np.max(mask_coverage_pcts):.1f}% (平均 {np.mean(mask_coverage_pcts):.1f}%)')
print(f'{"="*50}')

## Step 3：合併所有資料集（Roboflow + Urban Street + Xiang Tree）

將 **8 個資料來源** 合併到 `/content/merged_dataset/`，自動：
1. 讀取 Roboflow 資料集的 `data.yaml` 找出 trunk class ID
2. 加入已轉換的 Urban Street 資料（已是 YOLO-seg 格式，class 0）
3. 加入 Xiang et al. 訓練子集（80%，SAM 標註，class 0）
4. 只保留 trunk 類別的標註（移除 post/sprinkler 等）
5. 影像快速去重（檔案大小 + 前 2KB hash，避免重複）
6. 學術資料加權：Urban Street 2x、Xiang 3x，保留 Roboflow 多樣性 ~13%
7. 重新分配 train/valid/test = 80%/15%/5%

In [ ]:
import yaml, os, shutil, hashlib, random
from pathlib import Path
from collections import defaultdict

MERGED_DIR = '/content/merged_dataset'
URBAN_CONVERTED_DIR = '/content/urban_street_converted'

# 清理舊的合併目錄
if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)

for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(MERGED_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(MERGED_DIR, split, 'labels'), exist_ok=True)

# ===== 從每個 Roboflow 資料集提取 trunk-only 的 (image, label) 對 =====
all_pairs = []  # [(img_path, label_lines), ...]
seen_hashes = set()
stats = defaultdict(int)

def fast_file_hash(filepath):
    """快速去重 hash：檔案大小 + 前 2KB，比 MD5 快 50-100x"""
    import hashlib
    st = os.stat(filepath)
    with open(filepath, 'rb') as f:
        head = f.read(2048)
    return hashlib.md5(f'{st.st_size}:{head}'.encode('latin-1')).hexdigest()

print('='*50)
print('📦 Phase 1: 處理 Roboflow 資料集')
print('='*50)

for ds in downloaded_datasets:
    ds_dir = ds.location
    data_yaml = os.path.join(ds_dir, 'data.yaml')

    # 讀取 class 定義
    with open(data_yaml, 'r') as f:
        cfg = yaml.safe_load(f)
    names = cfg.get('names', [])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names.keys())]

    # ===== 嚴格過濾：只保留 trunk 類別 =====
    # 已知各資料集實際 class：
    #   bi-axe:     trunk, post, sprinkler      → 只要 trunk
    #   cherry:     trunk, post                 → 只要 trunk
    #   weppw:      trunk, 0, 1, 2, post, sprinkler → trunk + 0 + 2 是樹幹，1/post/sprinkler 排除
    #   z1bpo/ixblx: 下載後才知 class 名
    #
    # 特殊規則：weppw 資料集中 class "0" 和 "2" 經確認也是樹幹標註
    EXTRA_TRUNK_OVERRIDES = {
        'tree_trunk-weppw': {'0', '2'},  # 這兩個 class 經人工確認為樹幹
    }

    ds_basename = os.path.basename(ds_dir)
    extra_trunk = set()
    for key, val in EXTRA_TRUNK_OVERRIDES.items():
        if key in ds_basename:
            extra_trunk = val
            break

    trunk_ids = set()
    non_trunk_names = []
    for idx, name in enumerate(names):
        n = str(name).lower().strip()
        # 嚴格匹配：class name 包含 'trunk'，或在手動確認的 override 清單中
        if 'trunk' in n or n in extra_trunk:
            trunk_ids.add(idx)
        else:
            non_trunk_names.append(f'{idx}:{name}')

    if trunk_ids:
        override_note = ''
        if extra_trunk:
            override_note = f' (含手動確認的 trunk class: {extra_trunk})'
        print(f'  ✓ {ds_basename}: 保留 trunk class IDs = {trunk_ids}{override_note}')
        if non_trunk_names:
            print(f'    ⛔ 已排除非樹幹類別: {non_trunk_names}')
    elif len(names) == 1:
        # 單一類別資料集：如果 class name 不含 trunk 但整個專案是 trunk segmentation，
        # 合理假設這唯一的 class 就是 trunk
        trunk_ids = {0}
        print(f'  ⚠️ {ds_basename}: 單類別 "{names[0]}"，假設為 trunk')
    else:
        # 多類別但沒有任何 class 含 'trunk'  → 跳過整個資料集！
        print(f'  ❌ {ds_basename}: {len(names)} 個類別都不含 "trunk"!')
        print(f'     classes = {names}')
        print(f'     整個資料集跳過，避免引入錯誤標註')
        stats['datasets_skipped'] += 1
        continue  # 跳過此資料集

    # 遍歷所有 splits
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(ds_dir, split, 'images')
        lbl_dir = os.path.join(ds_dir, split, 'labels')
        if not os.path.exists(img_dir):
            continue

        for img_file in os.listdir(img_dir):
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                continue
            img_path = os.path.join(img_dir, img_file)

            # 去重：快速 hash（檔案大小 + 前 2KB，比全檔 MD5 快 50-100x）
            img_hash = fast_file_hash(img_path)
            if img_hash in seen_hashes:
                stats['duplicates'] += 1
                continue
            seen_hashes.add(img_hash)

            # 讀取標籤並過濾
            stem = os.path.splitext(img_file)[0]
            lbl_path = os.path.join(lbl_dir, stem + '.txt')
            trunk_lines = []
            if os.path.exists(lbl_path):
                with open(lbl_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            cls = int(parts[0])
                            if cls in trunk_ids:
                                parts[0] = '0'  # 統一為 class 0
                                trunk_lines.append(' '.join(parts))
                                stats['annotations_kept'] += 1
                            else:
                                stats['annotations_removed'] += 1

            if trunk_lines:
                all_pairs.append((img_path, trunk_lines))
                stats['images_with_trunk'] += 1
            else:
                # 保留沒有 trunk 的影像作為負樣本（少量）
                if random.random() < 0.1:  # 10% 機率保留負樣本
                    all_pairs.append((img_path, []))
                    stats['negative_samples'] += 1
                else:
                    stats['images_no_trunk_skipped'] += 1

roboflow_count = len(all_pairs)
print(f'\n  Roboflow 影像: {roboflow_count}')

# ===== Phase 2: 加入 Urban Street 已轉換的資料 =====
print(f'\n{"="*50}')
print('📦 Phase 2: 處理 Urban Street Trunk 資料集')
print('='*50)

urban_img_dir = os.path.join(URBAN_CONVERTED_DIR, 'images')
urban_lbl_dir = os.path.join(URBAN_CONVERTED_DIR, 'labels')

if os.path.exists(urban_img_dir):
    urban_images = [f for f in os.listdir(urban_img_dir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    print(f'  找到 {len(urban_images)} 張已轉換的 Urban Street 圖片')

    urban_added = 0
    for img_file in urban_images:
        img_path = os.path.join(urban_img_dir, img_file)

        # 去重（快速 hash）
        img_hash = fast_file_hash(img_path)
        if img_hash in seen_hashes:
            stats['duplicates'] += 1
            continue
        seen_hashes.add(img_hash)

        # 讀取已轉換的 YOLO 標籤（已經是 class 0）
        stem = os.path.splitext(img_file)[0]
        lbl_path = os.path.join(urban_lbl_dir, stem + '.txt')
        trunk_lines = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    line = line.strip()
                    if line and len(line.split()) >= 5:
                        trunk_lines.append(line)
                        stats['annotations_kept'] += 1

        if trunk_lines:
            all_pairs.append((img_path, trunk_lines))
            stats['images_with_trunk'] += 1
            urban_added += 1
        else:
            if random.random() < 0.05:  # 5% 負樣本
                all_pairs.append((img_path, []))
                stats['negative_samples'] += 1

    print(f'  ✓ Urban Street 加入: {urban_added} 張')
else:
    print('  ⚠️ 未找到 Urban Street 轉換資料，跳過')
    print(f'    （預期路徑: {URBAN_CONVERTED_DIR}）')
    print(f'    如果不需要 Urban Street 資料，可以繼續執行')

# ===== Phase 2.5: 加入 Xiang et al. 訓練子集 =====
print(f'\n{"="*50}')
print('📦 Phase 2.5: 處理 Xiang et al. 樹木訓練資料')
print('='*50)

XIANG_TRAIN_DIR = '/content/xiang_yolo_train'
xiang_train_img_dir = os.path.join(XIANG_TRAIN_DIR, 'images')
xiang_train_lbl_dir = os.path.join(XIANG_TRAIN_DIR, 'labels')

if os.path.exists(xiang_train_img_dir):
    xiang_images = [f for f in os.listdir(xiang_train_img_dir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    print(f'  找到 {len(xiang_images)} 張 Xiang 訓練圖片（SAM 標註）')

    xiang_added = 0
    for img_file in xiang_images:
        img_path = os.path.join(xiang_train_img_dir, img_file)

        # 去重（快速 hash）
        img_hash = fast_file_hash(img_path)
        if img_hash in seen_hashes:
            stats['duplicates'] += 1
            continue
        seen_hashes.add(img_hash)

        # 讀取已轉換的 YOLO 標籤（已經是 class 0）
        stem = os.path.splitext(img_file)[0]
        lbl_path = os.path.join(xiang_train_lbl_dir, stem + '.txt')
        trunk_lines = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    line = line.strip()
                    if line and len(line.split()) >= 5:
                        trunk_lines.append(line)
                        stats['annotations_kept'] += 1

        if trunk_lines:
            all_pairs.append((img_path, trunk_lines))
            stats['images_with_trunk'] += 1
            xiang_added += 1

    print(f'  ✓ Xiang tree 加入: {xiang_added} 張')
else:
    print('  ⚠️ 未找到 Xiang 訓練資料，跳過')
    print(f'    （預期路徑: {XIANG_TRAIN_DIR}）')
    print(f'    請先執行 Step 2.7')

# ===== Phase 3: 學術級資料 oversampling =====
# Urban Street = SCI 學術論文 7,675 張（29 樹種，專業標註）
# Xiang tree = SAM 標註 ~235 張（iPhone 近距離，真實樹木）
# 透過 oversampling 讓模型更重視高品質標註，但不過度壓縮 Roboflow 多樣性
# ⚠️ Roboflow 提供環境/光照/距離多樣性（巴西、不同角度、細瘦歪斜樹幹等）
#    如果學術資料加權太高，會壓縮這些多樣性的梯度貢獻，降低泛化能力
print(f'\n{"="*50}')
print('📦 Phase 3: 學術資料加權 (Oversampling)')
print('='*50)

URBAN_OVERSAMPLE = 1  # Urban Street 額外複製 1 次（原始 + 1 = 2x 權重）
                      # ⚡ Urban 已有 7,675 張 = Roboflow 的 3 倍，不需過度加權
XIANG_OVERSAMPLE = 2  # Xiang tree 額外複製 2 次（原始 + 2 = 3x 權重）
                      # 📱 量小但高度貼近 App 使用場景（iPhone 近距離），保留較高加權

# Urban Street oversampling
urban_pairs = [(p, l) for p, l in all_pairs if URBAN_CONVERTED_DIR in p]
urban_os_count = 0
for _ in range(URBAN_OVERSAMPLE):
    for pair in urban_pairs:
        all_pairs.append(pair)
        urban_os_count += 1

# Xiang tree oversampling
xiang_pairs = [(p, l) for p, l in all_pairs if XIANG_TRAIN_DIR in p]
xiang_os_count = 0
for _ in range(XIANG_OVERSAMPLE):
    for pair in xiang_pairs:
        all_pairs.append(pair)
        xiang_os_count += 1

print(f'  Urban Street:  {len(urban_pairs)} 原始 + {urban_os_count} 加權 = {len(urban_pairs) + urban_os_count} 有效')
print(f'  Xiang tree:    {len(xiang_pairs)} 原始 + {xiang_os_count} 加權 = {len(xiang_pairs) + xiang_os_count} 有效')
print(f'  Roboflow:      {roboflow_count} (1x，無加權)')
academic_total = len(urban_pairs) + urban_os_count + len(xiang_pairs) + xiang_os_count
print(f'  學術資料佔比:  {academic_total}/{len(all_pairs)} = {academic_total/len(all_pairs)*100:.1f}%')

# ===== 統計 =====
print(f'\n{"="*50}')
print(f'📊 合併統計')
print(f'{"="*50}')
print(f'  Roboflow 影像:      {roboflow_count}')
print(f'  Urban Street (2x):  {len(urban_pairs) + urban_os_count}')
print(f'  Xiang tree (3x):    {len(xiang_pairs) + xiang_os_count}')
print(f'  有效總計:           {len(all_pairs)}')
print(f'  重複影像跳過:       {stats["duplicates"]}')
print(f'  有 trunk 標註:      {stats["images_with_trunk"]}')
print(f'  負樣本保留:         {stats["negative_samples"]}')
print(f'  標註保留/移除:      {stats["annotations_kept"]} / {stats["annotations_removed"]}')
if stats['datasets_skipped'] > 0:
    print(f'  ⛔ 跳過的資料集:     {stats["datasets_skipped"]}（無 trunk 類別）')

# ===== 重新分配 train/valid/test =====
random.seed(42)
random.shuffle(all_pairs)

n = len(all_pairs)
n_train = int(n * 0.80)
n_valid = int(n * 0.15)

train_pairs = all_pairs[:n_train]
valid_pairs = all_pairs[n_train:n_train + n_valid]
test_pairs  = all_pairs[n_train + n_valid:]

print(f'\n🔄 寫入合併資料集到磁碟...')
for split_name, pairs in [('train', train_pairs), ('valid', valid_pairs), ('test', test_pairs)]:
    for i, (img_path, label_lines) in enumerate(pairs):
        if (i + 1) % 2000 == 0:
            print(f'  {split_name}: {i+1}/{len(pairs)}')
        ext = os.path.splitext(img_path)[1]
        # 用唯一編號避免檔名衝突
        new_name = f'{split_name}_{i:05d}'
        dst_img = os.path.join(MERGED_DIR, split_name, 'images', new_name + ext)
        dst_lbl = os.path.join(MERGED_DIR, split_name, 'labels', new_name + '.txt')
        # hard link 代替複製（瞬間完成，不佔磁碟空間）
        try:
            os.link(img_path, dst_img)
        except OSError:
            shutil.copy2(img_path, dst_img)  # fallback（跨檔案系統時）
        with open(dst_lbl, 'w') as f:
            f.write('\n'.join(label_lines) + '\n' if label_lines else '')

print(f'\n  Split 分配:')
print(f'    train: {len(train_pairs)}')
print(f'    valid: {len(valid_pairs)}')
print(f'    test:  {len(test_pairs)}')

# ===== 產生 data.yaml =====
data_yaml_path = os.path.join(MERGED_DIR, 'data.yaml')
data_config = {
    'path': MERGED_DIR,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['tree_trunk']
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f'\n✅ 合併完成！data.yaml: {data_yaml_path}')
print(f'   總影像數: {len(all_pairs)} 張（目標 10,000+）')
print(f'{"="*50}')

## Step 3.5：標註品質審核（每個資料集的 class 報告 + mask 統計）

⚠️ **在訓練之前必看！** 此 Cell 會：
1. 列出每個 Roboflow 資料集的 **所有 class 名稱** 和 **保留/排除** 情況
2. 分析每個 mask 的 **形狀特徵**（面積佔比、寬高比）— 檢查是否有 bounding-box 式的劣質標註
3. 從每個資料集各抽 2 張圖顯示標註比較 — 確認標註風格一致性

In [ ]:
#@title 🔍 標註品質審核 — 各資料集 class & mask 風格分析 { display-mode: "form" }
import yaml, os, cv2, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

print('='*60)
print('🔍 標註品質審核報告')
print('='*60)

# ===== Part 1: 每個 Roboflow 資料集的 class 報告 =====
print('\n📋 Part 1: 各資料集類別分析')
print('-'*60)

ds_audit = []  # [(data_dir, ds_name, classes_info)]

for i, ds in enumerate(downloaded_datasets):
    ds_dir = ds.location
    ds_name = os.path.basename(ds_dir)
    data_yaml = os.path.join(ds_dir, 'data.yaml')

    with open(data_yaml, 'r') as f:
        cfg = yaml.safe_load(f)
    names = cfg.get('names', [])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names.keys())]

    # 統計每個 class 的 annotation 數量
    class_counts = defaultdict(int)
    total_annotations = 0
    for split in ['train', 'valid', 'test']:
        lbl_dir = os.path.join(ds_dir, split, 'labels')
        if not os.path.exists(lbl_dir):
            continue
        for f in os.listdir(lbl_dir):
            if not f.endswith('.txt'):
                continue
            with open(os.path.join(lbl_dir, f), 'r') as fh:
                for line in fh:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        cls_name = names[cls_id] if cls_id < len(names) else f'unknown_{cls_id}'
                        class_counts[cls_name] += 1
                        total_annotations += 1

    # 判斷哪些 class 會被保留（與 Step 3 合併邏輯一致）
    EXTRA_TRUNK_OVERRIDES_AUDIT = {
        'tree_trunk-weppw': {'0', '2'},  # 手動確認為樹幹
    }
    extra_trunk = set()
    for key, val in EXTRA_TRUNK_OVERRIDES_AUDIT.items():
        if key in ds_name:
            extra_trunk = val
            break

    kept = []
    removed = []
    for idx, name in enumerate(names):
        n = str(name).lower().strip()
        count = class_counts.get(name, 0)
        if 'trunk' in n:
            kept.append(f'  ✓ [{idx}] {name}: {count} 個標註 → 保留 (含 trunk)')
        elif n in extra_trunk:
            kept.append(f'  ✓ [{idx}] {name}: {count} 個標註 → 保留 (手動確認為樹幹)')
        else:
            removed.append(f'  ✗ [{idx}] {name}: {count} 個標註 → 排除')

    print(f'\n📦 Dataset {i+1}: {ds_name}')
    print(f'   共 {len(names)} 個 class, {total_annotations} 個標註')
    for line in kept:
        print(line)
    for line in removed:
        print(line)
    if not kept:
        if len(names) == 1:
            print(f'  ⚠️ 單類別 "{names[0]}" — 假設為 trunk')
        else:
            print(f'  ❌ 無 trunk class → 整個資料集已跳過！')

    ds_audit.append((ds_dir, ds_name, names))

# ===== Part 2: Mask 形狀分析（檢查標註品質） =====
print(f'\n\n📐 Part 2: Mask 形狀分析（偵測劣質/矩形式標註）')
print('-'*60)

merged_train_imgs = sorted(Path(MERGED_DIR, 'train', 'images').glob('*'))
merged_train_lbls = [Path(str(p).replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt')
                      for p in merged_train_imgs]

# 分析 mask 的 面積佔比 和 凸包率（convexity）
aspect_ratios = []
area_ratios = []
polygon_point_counts = []
suspicious_count = 0

for lbl_path, img_path in zip(merged_train_lbls, merged_train_imgs):
    if not lbl_path.exists():
        continue
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        H, W = img.shape[:2]
    except:
        continue

    with open(lbl_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            coords = list(map(float, parts[1:]))
            n_points = len(coords) // 2
            polygon_point_counts.append(n_points)

            points = []
            for j in range(0, len(coords), 2):
                x = coords[j] * W
                y = coords[j+1] * H
                points.append([x, y])
            pts = np.array(points, dtype=np.float32)

            # bounding box
            x_min, y_min = pts.min(axis=0)
            x_max, y_max = pts.max(axis=0)
            bbox_w = x_max - x_min
            bbox_h = y_max - y_min

            if bbox_w > 0 and bbox_h > 0:
                ar = bbox_w / bbox_h
                aspect_ratios.append(ar)
                # mask 面積佔圖片比例
                mask_area = cv2.contourArea(pts.reshape(-1, 1, 2).astype(np.int32))
                img_area = W * H
                area_ratios.append(mask_area / img_area)

                # 凸包率：mask面積 / 外接矩形面積
                bbox_area = bbox_w * bbox_h
                fill_ratio = mask_area / bbox_area if bbox_area > 0 else 0
                # 如果 fill_ratio > 0.95 且 <6 個點 → 可能是 bounding box 式標註
                if fill_ratio > 0.95 and n_points <= 5:
                    suspicious_count += 1

total_masks = len(aspect_ratios)
print(f'\n  分析了 {total_masks} 個 mask:')
if total_masks > 0:
    print(f'    平均多邊形頂點數: {np.mean(polygon_point_counts):.1f} (越多 = 越精確)')
    print(f'    頂點數中位數:     {np.median(polygon_point_counts):.0f}')
    print(f'    低頂點 (≤4點):    {sum(1 for p in polygon_point_counts if p <= 4)} 個')
    print(f'    平均寬高比:       {np.mean(aspect_ratios):.3f} (樹幹應偏窄高, <1.0)')
    print(f'    面積佔比:         {np.mean(area_ratios)*100:.1f}% (樹幹通常 <15%)')
    print(f'    疑似矩形標註:      {suspicious_count} 個 (fill_ratio>0.95 且 ≤5點)')
    if suspicious_count > total_masks * 0.2:
        print(f'    ⚠️ 警告：{suspicious_count/total_masks*100:.0f}% 的標註疑似 bounding box 而非精確 polygon！')
    if np.mean(aspect_ratios) > 1.5:
        print(f'    ⚠️ 警告：平均寬高比偏高，可能標註了橫向物體（非直立樹幹）')

# ===== Part 3: 每資料來源各抽 2 張視覺化比較 =====
print(f'\n\n🎨 Part 3: 各資料來源標註風格比較')
print('-'*60)
print('  從每個資料集各抽 2 張圖，直觀比較標註風格')

def vis_with_mask(img_path, lbl_path):
    """回傳標註視覺化圖"""
    img = cv2.imread(str(img_path))
    if img is None:
        return np.zeros((300, 300, 3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    overlay = img.copy()
    n_masks = 0
    if os.path.exists(str(lbl_path)):
        with open(str(lbl_path), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 7:
                    continue
                coords = list(map(float, parts[1:]))
                points = []
                for j in range(0, len(coords), 2):
                    points.append([int(coords[j]*W), int(coords[j+1]*H)])
                pts = np.array(points, np.int32).reshape(-1, 1, 2)
                cv2.fillPoly(overlay, [pts], (0, 255, 0))
                cv2.polylines(img, [pts], True, (0, 255, 0), 2)
                n_masks += 1
    result = cv2.addWeighted(overlay, 0.3, img, 0.7, 0)
    return result

# 收集各來源的樣本
source_samples = defaultdict(list)  # source_name → [(img_path, lbl_path)]

# Roboflow 資料集
for i, ds in enumerate(downloaded_datasets):
    ds_name = os.path.basename(ds.location)
    for split in ['train', 'valid']:
        img_dir = os.path.join(ds.location, split, 'images')
        if not os.path.exists(img_dir):
            continue
        for f in os.listdir(img_dir):
            if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                img_p = os.path.join(img_dir, f)
                lbl_p = os.path.join(ds.location, split, 'labels',
                                    os.path.splitext(f)[0] + '.txt')
                if os.path.exists(lbl_p):
                    source_samples[f'RF{i+1}: {ds_name}'].append((img_p, lbl_p))

# Urban Street
urban_img_dir = os.path.join(URBAN_CONVERTED_DIR, 'images')
if os.path.exists(urban_img_dir):
    urban_lbl_dir = os.path.join(URBAN_CONVERTED_DIR, 'labels')
    for f in os.listdir(urban_img_dir):
        if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp')):
            img_p = os.path.join(urban_img_dir, f)
            lbl_p = os.path.join(urban_lbl_dir, os.path.splitext(f)[0] + '.txt')
            if os.path.exists(lbl_p):
                source_samples['Urban Street'].append((img_p, lbl_p))

n_sources = len(source_samples)
if n_sources > 0:
    fig, axes = plt.subplots(n_sources, 2, figsize=(12, 4*n_sources))
    if n_sources == 1:
        axes = [axes]

    for row, (source_name, pairs) in enumerate(sorted(source_samples.items())):
        sample_pairs = random.sample(pairs, min(2, len(pairs)))
        for col, (img_p, lbl_p) in enumerate(sample_pairs):
            ax = axes[row][col] if n_sources > 1 else axes[0][col]
            vis = vis_with_mask(img_p, lbl_p)
            ax.imshow(vis)
            ax.set_title(f'{source_name}\n{os.path.basename(img_p)}', fontsize=8)
            ax.axis('off')
        # If only 1 sample, blank the second column
        if len(sample_pairs) < 2:
            ax2 = axes[row][1] if n_sources > 1 else axes[0][1]
            ax2.axis('off')

    plt.suptitle('各資料來源的標註風格比較（綠色 = trunk mask）', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('\n👆 請確認：')
print('   1. 綠色 mask 是否緊貼樹幹邊緣（非整棵樹/樹冠）')
print('   2. 各資料集的標註風格是否大致一致')
print('   3. 如有劣質標註（矩形框/標錯物體），考慮移除該資料集')

## Step 4：查看幾張標註範例（確認標註品質）

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def visualize_yolo_seg_label(img_path, lbl_path):
    """視覺化 YOLO segmentation 標註"""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    
    overlay = img.copy()
    
    if lbl_path.exists():
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 7:  # class + at least 3 points (6 coords)
                    continue
                class_id = int(parts[0])
                coords = list(map(float, parts[1:]))
                
                # 反正規化：YOLO 存的是 0-1 比例,轉回像素
                points = []
                for i in range(0, len(coords), 2):
                    x = int(coords[i] * W)
                    y = int(coords[i+1] * H)
                    points.append([x, y])
                
                pts = np.array(points, np.int32).reshape((-1, 1, 2))
                
                # 半透明綠色填充 = 樹幹區域
                cv2.fillPoly(overlay, [pts], (0, 255, 0))
                # 綠色邊框
                cv2.polylines(img, [pts], True, (0, 255, 0), 2)
    
    # 混合
    result = cv2.addWeighted(overlay, 0.3, img, 0.7, 0)
    return result

# 隨機顯示 8 張標註範例（從合併後的 train set）
import random
train_imgs = sorted(Path(MERGED_DIR, 'train', 'images').glob('*'))
samples = random.sample(train_imgs, min(8, len(train_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
for ax, img_path in zip(axes.flat, samples):
    lbl_path = Path(str(img_path).replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt')
    vis = visualize_yolo_seg_label(img_path, lbl_path)
    ax.imshow(vis)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')

plt.suptitle(f'合併資料集標註範例（綠色 = tree_trunk mask）— 共 {len(train_imgs)} 張 train', fontsize=16)
plt.tight_layout()
plt.show()

print('\n👆 確認綠色區域是否正確覆蓋樹幹')
print('   如果標註有問題，建議在 Roboflow 上修正後重新下載')

## Step 5：開始訓練！（CNN 加速優化 🚀）

支援兩種模式：
- **🆕 Fresh** — 從 COCO 預訓練的 YOLOv8-seg 做遷移學習
- **🔄 Resume** — 從你已有的 `.pt` 續練，用更低學習率保護已學特徵

### CNN 加速策略（不影響精度）
1. **RAM Cache** — 自動偵測可用 RAM，預載全部圖片消除 I/O 瓶頸
2. **cuDNN Benchmark** — 自動選最快的卷積演算法
3. **TF32**（Ampere+）— TensorFloat-32 模式，CNN 精度幾乎無損但速度快 2-3x
4. **Workers 提升** — 根據 GPU 等級增加 DataLoader 平行載入

### GPU 推薦（速度優先 + 加速功能利用率）
- **A100-80GB** → batch=40, workers=16 ⭐ **最推薦** — TF32 ✅ cuDNN ✅ AMP ✅ 全部用得上，~5-8hr
- **A100-40GB** → batch=16, workers=8 — 同架構但 VRAM 較小，~10-14hr
- **H100** (~79GB) → batch=40, workers=16 — 速度快 ~30% 但貴，FP8 Transformer Engine 對 CNN 無用
- **RTX PRO 6000** (96GB) → batch=48, workers=16 🚀 如果有的話最快
- **L4** (24GB) → batch=8, workers=8 — 預算方案，~18-25hr
- **T4** (16GB) → batch=4, workers=4 — 不推薦，太慢 ~35-50hr

In [ ]:
#@title 訓練參數設定（GPU 自動優化） { display-mode: "form" }
import torch

#@markdown ### 模型大小（僅在 fresh mode 時生效）
MODEL_SIZE = 'yolov8m-seg.pt' #@param ['yolov8n-seg.pt', 'yolov8s-seg.pt', 'yolov8m-seg.pt'] {type:"string"}
#@markdown - **n** (nano): 3.4M params, 最快, 適合手機 ⚡
#@markdown - **s** (small): 11.8M params, 更好 recall
#@markdown - **m** (medium): 25.9M params, 最準, RTX PRO 6000 推薦 ⭐
#@markdown ---
#@markdown ### 訓練參數（設 0 = 自動偵測 GPU 上限）
EPOCHS = 200 #@param {type:"slider", min:50, max:400, step:10}
BATCH_SIZE_OVERRIDE = 0 #@param [0, 8, 16, 32, 64] {type:"raw"}
IMAGE_SIZE = 960 #@param [320, 480, 640, 960] {type:"raw"}

# ===== GPU 自動偵測 & 最佳參數 =====
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'🖥️ GPU: {gpu_name}')
    print(f'   VRAM: {gpu_mem_gb:.1f} GB')

    # 根據 VRAM 自動設定最佳 batch size
    # 注意：validation 時 mask upsampling 需要額外 VRAM
    # batch 不能塞滿，要預留 ~15-20GB 給 val
    if gpu_mem_gb >= 90:       # RTX PRO 6000 Blackwell (97GB)
        auto_batch = 32 if IMAGE_SIZE >= 960 else 64
        gpu_tier = 'RTX PRO 6000'
    elif gpu_mem_gb >= 70:     # H100 (~79GB) / A100-80G
        auto_batch = 24 if IMAGE_SIZE >= 960 else 48
        gpu_tier = 'H100/A100-80G'
    elif gpu_mem_gb >= 35:     # A100 (40GB)
        auto_batch = 12 if IMAGE_SIZE >= 960 else 24
        gpu_tier = 'A100-40G'
    elif gpu_mem_gb >= 20:     # L4 (24GB) / RTX 4090
        auto_batch = 6 if IMAGE_SIZE >= 960 else 12
        gpu_tier = 'L4/RTX4090'
    else:                      # T4 (16GB)
        auto_batch = 4 if IMAGE_SIZE >= 960 else 8
        gpu_tier = 'T4'

    BATCH_SIZE = BATCH_SIZE_OVERRIDE if BATCH_SIZE_OVERRIDE > 0 else auto_batch

    # ⚠️ workers=0 是關鍵！在 Colab 環境中 multiprocessing workers 容易死鎖
    WORKERS = 0

    # ⚠️ cache=True 在 ultralytics 裡其實是 RAM cache！
    #    必須用字串 'disk' 才是真正的磁碟快取
    cache_mode = 'disk'
    cache_note = 'Disk 快取 (穩定模式，避免 RAM OOM)'

    print(f'   Tier: {gpu_tier}')
    print(f'   Batch: {BATCH_SIZE} ({"自動" if BATCH_SIZE_OVERRIDE == 0 else "手動覆蓋"})')
    print(f'   Workers: {WORKERS} (防死鎖模式)')
    print(f'   Cache: {cache_note}')
else:
    print('⚠️ 沒有偵測到 GPU！請確認已選擇 GPU 執行階段')
    BATCH_SIZE = 8
    WORKERS = 0
    cache_mode = 'disk'

# ===== 根據模式決定起始模型 & 學習率 =====
if TRAINING_MODE == 'resume' and RESUME_MODEL_PATH:
    start_model = RESUME_MODEL_PATH
    LR0 = 0.0003
    LRF = 0.005
    WARMUP = 3
    PATIENCE = 50
    print(f'\n🔄 續練模式 (Resume)')
    print(f'   基礎模型: {RESUME_MODEL_PATH}')
    print(f'   學習率: lr0={LR0} (低，保護已學特徵)')
else:
    start_model = MODEL_SIZE
    LR0 = 0.001
    LRF = 0.01
    WARMUP = 5
    PATIENCE = 40
    params_info = {'yolov8n-seg.pt': '3.4M', 'yolov8s-seg.pt': '11.8M', 'yolov8m-seg.pt': '25.9M'}

    # --- 自動偵測 crash 留下的 last.pt ---
    if AUTO_RESUME_FROM_CRASH:
        _crash_candidates = [
            '/content/runs/segment/train/weights/last.pt',
            '/content/runs/segment/train2/weights/last.pt',
            '/content/runs/segment/train3/weights/last.pt',
        ]
        for _cp in _crash_candidates:
            if os.path.exists(_cp):
                start_model = _cp
                LR0 = 0.0005   # 已有不錯的權重，用較小 lr 繼續
                WARMUP = 2
                print(f'\n🔁 偵測到上次 crash 的 checkpoint: {_cp}')
                print(f'   自動從該權重繼續訓練（不是從 epoch 0 COCO 重新來）')
                print(f'   學習率: lr0={LR0} (降低以保護已學特徵)')
                break
        else:
            print(f'\n🆕 全新訓練 (Fresh)')
            print(f'   基礎模型: {MODEL_SIZE} ({params_info.get(MODEL_SIZE, "?")} params)')
            print(f'   學習率: lr0={LR0}')
    else:
        print(f'\n🆕 全新訓練 (Fresh)')
        print(f'   基礎模型: {MODEL_SIZE} ({params_info.get(MODEL_SIZE, "?")} params)')
        print(f'   學習率: lr0={LR0}')

print(f'\n📋 訓練配置:')
print(f'   Epochs:  {EPOCHS}')
print(f'   Batch:   {BATCH_SIZE}')
print(f'   ImgSz:   {IMAGE_SIZE}')
print(f'   Workers: {WORKERS} (防死鎖)')
print(f'   Images:  ~{len(all_pairs)} 張')

In [ ]:
from ultralytics import YOLO
import time, torch

# === CNN 加速設定（不影響精度）===
# 1. cuDNN benchmark: 自動選最快的卷積演算法
# 2. TF32: Ampere+ GPU 的 TensorFloat-32，CNN 精度幾乎無損但速度快 2-3x
torch.backends.cudnn.benchmark = True
if hasattr(torch.backends, 'cuda'):
    torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.allow_tf32 = True

# 載入起始模型
model = YOLO(start_model)
mode_label = '續練' if (TRAINING_MODE == 'resume' and RESUME_MODEL_PATH) else '全新訓練'
print(f'✓ 載入模型: {start_model}')
print(f'  模式: {mode_label}')
print(f'  Training images: ~{len(train_pairs)}')
print(f'  Batch: {BATCH_SIZE} | Workers: {WORKERS} | ImgSz: {IMAGE_SIZE}')
print(f'  Cache: {cache_mode} | cuDNN benchmark: ON | TF32: ON')
print()

start_time = time.time()

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMAGE_SIZE,
    device=0,               # GPU 0
    workers=WORKERS,         # 根據 GPU tier 自動調整
    cache=cache_mode,        # RAM 快取消除 I/O 瓶頸
    verbose=True,            # 顯示每 epoch 進度條
    patience=PATIENCE,
    save=True,
    save_period=10,
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=LR0,                # 續練模式用更低的學習率
    lrf=LRF,
    warmup_epochs=WARMUP,
    cos_lr=True,
    
    # 資料增強（針對戶外樹木場景 + 細瘦/歪斜樹幹加強）
    hsv_h=0.02,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=15.0,            # 增加旋轉 (10→15)，對應傾斜/彎曲樹幹
    translate=0.15,
    scale=0.5,               # 增加縮放範圍 (0.4→0.5)，對應遠近不同粗細
    shear=3.0,               # 增加切變 (2→3)，模擬不同拍攝角度
    perspective=0.0005,
    flipud=0.0,              # 不上下翻轉
    fliplr=0.5,
    mosaic=0.9,
    mixup=0.15,
    copy_paste=0.15,         # 增加複製貼上 (0.1→0.15)，更多合成實例
    erasing=0.15,
    
    # 分割設定（mask_ratio=1 + imgsz=960 → 最高精度 mask）
    overlap_mask=True,
    mask_ratio=1,
    
    amp=True,
    close_mosaic=15,
)

elapsed = time.time() - start_time
print(f'\n✓ 訓練完成！耗時 {elapsed/60:.1f} 分鐘')
print(f'  模式: {mode_label}')
print(f'  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## Step 6：驗證模型品質

In [ ]:
from ultralytics import YOLO
import os

# 找到最佳模型
best_pt = '/content/runs/segment/train/weights/best.pt'
if not os.path.exists(best_pt):
    # 嘗試其他路徑
    for p in ['/content/runs/segment/train2/weights/best.pt',
              '/content/runs/segment/train3/weights/best.pt']:
        if os.path.exists(p):
            best_pt = p
            break

print(f'Model: {best_pt}')
size_mb = os.path.getsize(best_pt) / 1024 / 1024
print(f'Size: {size_mb:.1f} MB')

# 驗證
model = YOLO(best_pt)
metrics = model.val(data=data_yaml_path)

box_map50 = float(getattr(metrics.box, 'map50', 0))
seg_map50 = float(getattr(metrics.seg, 'map50', 0))
box_p = float(getattr(metrics.box, 'mp', 0))
box_r = float(getattr(metrics.box, 'mr', 0))

print(f'\n{"="*50}')
print(f'  Detection  mAP50: {box_map50:.3f}  P: {box_p:.3f}  R: {box_r:.3f}')
print(f'  Segment    mAP50: {seg_map50:.3f}')
print(f'{"="*50}')

if seg_map50 >= 0.85:
    print('\n✅ 模型品質優秀！可以匯出到手機')
elif seg_map50 >= 0.70:
    print('\n⚠️ 模型品質可以，但建議加更多訓練資料')
else:
    print('\n❌ 品質不足，需要更多資料或更長訓練')

## Step 6.5：Xiang et al. 獨立分割品質驗證（~59 張驗證子集）

用 Xiang et al. 的 294 張中保留的 **20% 驗證子集**（~59 張）來測試模型的**分割品質**：
1. 跑 YOLO val → 取得 mAP50 / mAP50-95（Box & Mask）
2. 逐圖推論 → 偵測率、平均信心度、mask 覆蓋率統計
3. 隨機抽樣可視化 → 展示 mask overlay 結果

⚠️ 這些影像**不在訓練集中**（另外 80% 已加入訓練，這 20% 是完全獨立的）！
📸 iPhone 13 Pro 近距離拍攝（0.25~5m），SAM 標註的 trunk mask
📚 Xiang et al. (2025) Sensors, 25(16), 5060 — Zenodo DOI: 10.5281/zenodo.10650629

In [ ]:
#@title Xiang et al. 獨立分割品質驗證 { display-mode: "form" }
import os, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

XIANG_CONVERTED = '/content/xiang_yolo_validation'
xiang_yaml = os.path.join(XIANG_CONVERTED, 'data.yaml')
img_dir = os.path.join(XIANG_CONVERTED, 'images')

if not os.path.exists(xiang_yaml):
    print('⚠️ 未找到 Xiang et al. 驗證集，跳過此步驟')
    print('   請先執行 Step 2.7 下載 & 轉換')
else:
    # 統計影像數量
    all_img_files = sorted([
        f for f in os.listdir(img_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    n_val = len(all_img_files)

    print(f'📊 Xiang et al. 獨立分割品質驗證')
    print(f'   驗證子集影像: {n_val} 張（全部 294 張中的 20%，iPhone 近距離拍攝）')
    print(f'   ⚠️ 另外 80% 已加入訓練，這 {n_val} 張是完全獨立的\n')

    # ===== 1. YOLO val 指標 =====
    from ultralytics import YOLO
    val_model = YOLO(best_pt)

    print('1️⃣ YOLO 驗證指標 (mAP)...')
    xiang_seg_map50 = 0
    try:
        xiang_metrics = val_model.val(data=xiang_yaml, verbose=False)
        xiang_seg_map50 = float(getattr(xiang_metrics.seg, 'map50', 0))
        xiang_seg_map5095 = float(getattr(xiang_metrics.seg, 'map', 0))
        xiang_box_map50 = float(getattr(xiang_metrics.box, 'map50', 0))
        xiang_box_map5095 = float(getattr(xiang_metrics.box, 'map', 0))
        print(f'   Box  mAP50:    {xiang_box_map50:.3f}')
        print(f'   Box  mAP50-95: {xiang_box_map5095:.3f}')
        print(f'   Mask mAP50:    {xiang_seg_map50:.3f}')
        print(f'   Mask mAP50-95: {xiang_seg_map5095:.3f}')
    except Exception as e:
        print(f'   ⚠️ 驗證指標計算失敗: {e}')

    # ===== 2. 逐圖推論 → 統計 =====
    print(f'\n2️⃣ 逐圖推論 → 統計...')
    all_results = []

    for img_file in all_img_files:
        img_path = os.path.join(img_dir, img_file)

        # 推論
        preds = val_model.predict(img_path, verbose=False)
        n_det = len(preds[0].boxes)
        max_conf = float(preds[0].boxes.conf.max()) if n_det > 0 else 0.0

        # Mask 面積 (像素數)
        mask_area_px = 0
        if preds[0].masks is not None and n_det > 0:
            best_idx = int(preds[0].boxes.conf.argmax())
            mask_data = preds[0].masks.data[best_idx].cpu().numpy()
            mask_area_px = int((mask_data > 0.5).sum())

        all_results.append({
            'file': img_file,
            'detected': n_det > 0,
            'conf': max_conf,
            'n_det': n_det,
            'mask_area_px': mask_area_px,
        })

    # ===== 3. 統計報告 =====
    print(f'📊 Xiang et al. 獨立分割品質報告（{n_val} 張驗證子集）')
    print(f'📊 Xiang et al. 獨立分割品質報告（294 張真實樹木）')
    print(f'{"="*60}')

    n_total = len(all_results)
    n_detected = sum(1 for x in all_results if x['detected'])
    det_rate = n_detected / n_total * 100 if n_total > 0 else 0
    confs = [x['conf'] for x in all_results if x['detected']]
    avg_conf = np.mean(confs) if confs else 0
    areas = [x['mask_area_px'] for x in all_results if x['mask_area_px'] > 0]

    print(f'\n  偵測率:      {n_detected}/{n_total} = {det_rate:.1f}%')
    print(f'  平均信心度:  {avg_conf:.3f}')
    if areas:
        print(f'  Mask 面積:   {np.mean(areas):.0f} px (平均)')
        print(f'  面積範圍:    {min(areas)} ~ {max(areas)} px')
        print(f'  面積中位數:  {np.median(areas):.0f} px')

    # 失敗案例
    failed = [x['file'] for x in all_results if not x['detected']]
    if failed:
        print(f'\n  ❌ 未偵測 ({len(failed)} 張):')
        for f in failed[:10]:
            print(f'    {f}')
        if len(failed) > 10:
            print(f'    ... 還有 {len(failed)-10} 張')

    # 信心度分佈
    if confs:
        print(f'\n  📊 信心度分佈:')
        for threshold in [0.9, 0.8, 0.7, 0.5, 0.3]:
            cnt = sum(1 for c in confs if c >= threshold)
            print(f'    ≥ {threshold:.1f}: {cnt}/{len(confs)} ({cnt/len(confs)*100:.1f}%)')

    # 品質評級
    print(f'\n  📋 品質評級:')
    if xiang_seg_map50 >= 0.85 and det_rate >= 95:
        print(f'  🏆 優秀 — 獨立驗證 mAP50={xiang_seg_map50:.3f}, 偵測率={det_rate:.1f}%')
    elif xiang_seg_map50 >= 0.70 and det_rate >= 85:
        print(f'  ✅ 良好 — 獨立驗證 mAP50={xiang_seg_map50:.3f}, 偵測率={det_rate:.1f}%')
    else:
        print(f'  ⚠️ 待加強 — 獨立驗證 mAP50={xiang_seg_map50:.3f}, 偵測率={det_rate:.1f}%')

    print(f'{"="*60}')

    # ===== 4. 可視化 =====
    print(f'\n3️⃣ 隨機抽樣可視化 (6 張)...')
    detected_items = [x for x in all_results if x['detected']]
    samples = random.sample(detected_items, min(6, len(detected_items)))

    n_samples = len(samples)
    cols = min(3, n_samples)
    rows = (n_samples + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 6*rows))
    if rows * cols == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)
    fig.suptitle('Xiang et al. 獨立驗證 — 真實樹幹分割結果 (mask overlay)', fontsize=16)

    for idx, item in enumerate(samples):
        r, c = divmod(idx, cols)
        ax = axes[r, c]
        img_path = os.path.join(img_dir, item['file'])
        preds = val_model.predict(img_path, verbose=False)
        annotated = preds[0].plot()
        ax.imshow(annotated[:, :, ::-1])
        ax.set_title(
            f'conf={item["conf"]:.2f} | area={item["mask_area_px"]}px',
            fontsize=9
        )
        ax.axis('off')

    # 隱藏空白 axes
    for idx in range(n_samples, rows * cols):
        r, c = divmod(idx, cols)
        axes[r, c].axis('off')

    plt.tight_layout()
    plt.show()

## Step 7：看看模型在真實圖片上的表現

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import random

# 在驗證集上跑推論
val_imgs = sorted(Path(MERGED_DIR, 'valid', 'images').glob('*'))
if not val_imgs:
    val_imgs = sorted(Path(MERGED_DIR, 'test', 'images').glob('*'))

samples = random.sample(val_imgs, min(8, len(val_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
for ax, img_path in zip(axes.flat, samples):
    results = model.predict(str(img_path), verbose=False)
    annotated = results[0].plot()  # 畫出 bbox + mask
    ax.imshow(annotated[:, :, ::-1])  # BGR → RGB
    
    n_det = len(results[0].boxes)
    confs = [f'{c:.2f}' for c in results[0].boxes.conf.cpu().numpy()] if n_det > 0 else []
    ax.set_title(f'{img_path.name}\n{n_det} det {confs}', fontsize=8)
    ax.axis('off')

plt.suptitle('模型推論結果（mask = 偵測到的樹幹）', fontsize=16)
plt.tight_layout()
plt.show()

## Step 8：匯出模型

匯出三種格式：
- **TFLite INT8**：手機端 Flutter 使用（~6.3 MB）
- **ONNX**：伺服器端使用
- **PyTorch**：備份

In [ ]:
import shutil

export_dir = '/content/exported_models'
os.makedirs(export_dir, exist_ok=True)

# ===== 匯出尺寸策略 =====
# 訓練用 960 獲得最精準的 mask 權重
# 手機端 TFLite 用 640 保持流暢 FPS（30+ FPS）
# 伺服器端 ONNX 用 960 保持最高精度
TFLITE_EXPORT_SIZE = 640   # 手機端：速度優先
ONNX_EXPORT_SIZE = IMAGE_SIZE  # 伺服器端：精度優先

# 1. TFLite INT8（手機端，640）
print(f'匯出 TFLite INT8 (imgsz={TFLITE_EXPORT_SIZE}, 手機端用)...')
try:
    tflite_path = model.export(format='tflite', imgsz=TFLITE_EXPORT_SIZE, int8=True)
    if tflite_path:
        shutil.copy2(tflite_path, os.path.join(export_dir, 'tree_trunk_seg.tflite'))
        size = os.path.getsize(tflite_path) / 1024 / 1024
        print(f'  ✓ TFLite: {size:.1f} MB (input={TFLITE_EXPORT_SIZE})')
except Exception as e:
    print(f'  ✗ TFLite export failed: {e}')
    print('  Trying FP16 fallback...')
    try:
        tflite_path = model.export(format='tflite', imgsz=TFLITE_EXPORT_SIZE, half=True)
        if tflite_path:
            shutil.copy2(tflite_path, os.path.join(export_dir, 'tree_trunk_seg.tflite'))
            size = os.path.getsize(tflite_path) / 1024 / 1024
            print(f'  ✓ TFLite FP16: {size:.1f} MB (input={TFLITE_EXPORT_SIZE})')
    except Exception as e2:
        print(f'  ✗ TFLite FP16 also failed: {e2}')

# 2. ONNX（伺服器端，960）
print(f'\n匯出 ONNX (imgsz={ONNX_EXPORT_SIZE}, 伺服器端用)...')
try:
    onnx_path = model.export(format='onnx', imgsz=ONNX_EXPORT_SIZE, simplify=True)
    if onnx_path:
        shutil.copy2(onnx_path, os.path.join(export_dir, 'tree_trunk_seg.onnx'))
        size = os.path.getsize(onnx_path) / 1024 / 1024
        print(f'  ✓ ONNX: {size:.1f} MB (input={ONNX_EXPORT_SIZE})')
except Exception as e:
    print(f'  ✗ ONNX export failed: {e}')

# 3. 複製 PyTorch best.pt
shutil.copy2(best_pt, os.path.join(export_dir, 'tree_trunk_seg_best.pt'))
print(f'\n✓ PyTorch: {os.path.getsize(best_pt) / 1024 / 1024:.1f} MB')

# Labels
with open(os.path.join(export_dir, 'tree_trunk_labels.txt'), 'w') as f:
    f.write('tree_trunk\n')
print('✓ Labels file created')

print(f'\n所有模型已匯出到: {export_dir}')
!ls -la {export_dir}

## Step 9：下載模型

打包所有模型成 zip，點擊下載到你的電腦。

下載後：
1. 把 `tree_trunk_seg.tflite` 放到 `frontend/assets/ml/`
2. 把 `tree_trunk_labels.txt` 放到 `frontend/assets/ml/`
3. 把 `tree_trunk_seg.onnx` 放到 `backend/ml_service/models/`（伺服器端用）

In [ ]:
# 打包下載
!cd /content && zip -r exported_models.zip exported_models/

from google.colab import files
files.download('/content/exported_models.zip')
print('\n✓ 下載開始！')
print('\n下載後的部署步驟：')
print('  1. 解壓縮')
print('  2. tree_trunk_seg.tflite → frontend/assets/ml/')
print('  3. tree_trunk_labels.txt → frontend/assets/ml/')
print('  4. tree_trunk_seg.onnx   → backend/ml_service/models/')
print('  5. tree_trunk_seg_best.pt → 保留備份（日後可繼續訓練）')

## Step 10（可選）：用你自己的照片測試

In [ ]:
#@title 上傳照片測試 { display-mode: "form" }
from google.colab import files
import cv2
import matplotlib.pyplot as plt

print('請上傳一張樹幹照片...')
uploaded = files.upload()

for filename, data in uploaded.items():
    # 存檔
    with open(f'/content/{filename}', 'wb') as f:
        f.write(data)
    
    # 推論
    results = model.predict(f'/content/{filename}', verbose=False)
    annotated = results[0].plot()
    
    n_det = len(results[0].boxes)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated[:, :, ::-1])
    plt.title(f'偵測結果：{n_det} 個樹幹', fontsize=16)
    plt.axis('off')
    plt.show()
    
    if n_det > 0:
        for i, (box, conf) in enumerate(zip(results[0].boxes.xyxy, results[0].boxes.conf)):
            print(f'  樹幹 {i+1}: bbox={box.cpu().numpy().astype(int).tolist()} conf={conf:.3f}')
        if results[0].masks is not None:
            print(f'  Mask pixels: {int(results[0].masks.data.sum())}')
    else:
        print('  未偵測到樹幹')

---
## 訓練完成！

### 這次用了什麼？
- 合併 **8 個資料來源**（6 Roboflow + 1 Kaggle + 1 Zenodo 學術級），約 **10,000+ 張** → 學術加權後有效 **~18,500+ 張**
- **Urban Street Trunk**：浙江農林大學 SCI 論文，7,675 張城市行道樹（VOC → YOLO 自動轉換，2x 加權）
- **Xiang et al. 訓練子集**：~235 張真實樹木（Purdue 校園, iPhone 13 Pro, SAM 標註，3x 加權）+ ~59 張獨立驗證
- 只保留 `tree_trunk` 類別，29 種樹幹全部合併為單一類別
- YOLOv8m-seg（25.9M 參數）Fresh 訓練，imgsz=960, mask_ratio=1
- GPU 自動偵測：RTX PRO 6000 / H100 (~79GB) / A100，自動用滿算力
- 強化資料增強（degrees=15, scale=0.5, copy-paste=0.15, shear=3.0）→ 提升細瘦/歪斜樹幹通用性
- 支援從已有 `.pt` 模型續練（Resume mode）

### 你得到了什麼？
- `tree_trunk_seg.tflite` — 手機端模型（INT8 量化, imgsz=640, ~6-12 MB, 30+ FPS）
- `tree_trunk_seg.onnx` — 伺服器端模型（imgsz=960, 最高精度）
- `tree_trunk_seg_best.pt` — PyTorch 備份（可以繼續訓練）

### 接下來？
1. 下載模型到電腦
2. 放到對應的目錄
3. 更新 Flutter `tflite_tracking_service.dart` 的推論程式碼
4. 查看 Step 6.5 的獨立分割品質驗證結果，確認模型在 Xiang et al. 獨立驗證子集上的分割表現

### 想要更準？
- 加入你自己在台灣拍的樹幹照片（用 Roboflow 標註）
- 在 `DATASETS` 清單加入更多資料集
- 拿這次的 best.pt 用 **Resume 模式** 再訓練一次